# Section 1: Importing Nemosis Library and Fetching AEMO Data
Import required libraries and configure nemosis to download generator registration and dispatch unit SCADA data from AEMO for NSW renewable sources.

In [ ]:
# Core libraries for data handling and plotting
import pandas as pd          # dataframe operations
import numpy as np           # numerical utilities
from datetime import timedelta  # time arithmetic

# Nemosis helpers for fetching AEMO static and dynamic data
from nemosis import static_table, dynamic_data_compiler
import nemosis.defaults as defaults

from pathlib import Path      # filesystem paths
import plotly.express as px   # interactive plotting

In [ ]:
# Path where downloaded/raw AEMO files will be cached locally
raw_data_cache = 'C:\\Users\\Sankalp Roy\\Documents\\BACapstoneProject\\workshop_coding_files\\raw_data_cache'

In [ ]:
# Ensure the local raw data folder is used
raw_data_cache = "raw_data_cache"

# Point nemosis at the official generator registration spreadsheet
defaults.static_table_url["Generators and Scheduled Loads"] = (
    "https://www.aemo.com.au/-/media/files/electricity/nem/participant_information/"
    "nem-registration-and-exemption-list.xlsx"
    "?rev=9563c1bc8ccb4ad19e429d5ab767062c&sc_lang=en"
)

# Remove problematic older Excel file if present
bad_file = Path(raw_data_cache) / "NEM Registration and Exemption List.xls"
if bad_file.exists():
    bad_file.unlink()

# Load the generators and scheduled loads static table (caches to raw_data_cache)
dispatch_units = static_table(
    table_name="Generators and Scheduled Loads",
    raw_data_location=raw_data_cache,
    update_static_file=True,
)

# Show a preview of the loaded table
dispatch_units.head()

# Section 2: Processing 5-Minute Interval Renewable Generation Data
Extract and aggregate 5-minute interval SCADA dispatch values for Solar, Wind, and Hydro generators operating in NSW1 over the 2021-2024 period.

In [ ]:
# Extract list of DUIDs for renewable generators (Solar, Wind, Hydro)
renewable_duids = dispatch_units.loc[
    dispatch_units["Fuel Source - Primary"].isin(["Solar", "Wind", "Hydro"]),
    "DUID"
]

# Convert selection to a plain Python list for filtering later
renewable_duid_list = renewable_duids.tolist()
renewable_duid_list[:10]

In [ ]:
# Filter renewable generators to those located in NSW1
nsw_renewable_generators = dispatch_units.loc[
    (dispatch_units["Fuel Source - Primary"].isin(["Solar", "Wind", "Hydro"])) &
    (dispatch_units["Region"] == "NSW1"),
    ["DUID", "Region", "Fuel Source - Primary"]
]

# Show the filtered table
nsw_renewable_generators

In [ ]:
# Get unique DUIDs for NSW renewable generators
nsw_renewable_duids = nsw_renewable_generators["DUID"].dropna().unique().tolist()

# Download/compile SCADA dispatch values for those DUIDs over the 2021-07-01 to 2024-07-01 window
scada_data = dynamic_data_compiler(
    start_time="2021/07/01 00:00:00",
    end_time="2024/07/01 00:00:00",  # covers up to 30 June 2024
    table_name="DISPATCH_UNIT_SCADA",
    raw_data_location=raw_data_cache,
    select_columns=["SETTLEMENTDATE", "DUID", "SCADAVALUE"],
    filter_cols=["DUID"],
    filter_values=(nsw_renewable_duids,),
    fformat="parquet",
)

# Preview downloaded SCADA rows
scada_data.head()

In [ ]:
# Convert settlement timestamps to datetime and drop any rows from July 2024
scada_data["SETTLEMENTDATE"] = pd.to_datetime(scada_data["SETTLEMENTDATE"])

scada_data = scada_data[
    (scada_data["SETTLEMENTDATE"] >= "2021-07-01") &
    (scada_data["SETTLEMENTDATE"] < "2024-07-01")
]

In [ ]:
# Export SCADA data snapshot to CSV for offline inspection
scada_data.to_csv("C:\\Users\\Sankalp Roy\\Documents\\BACapstoneProject\\workshop_coding_files\\Scada_2021_2024.csv", index=False)

In [ ]:
# Map each DUID to its primary fuel source and join to SCADA values
duid_fuel_map = nsw_renewable_generators[
    ["DUID", "Fuel Source - Primary"]
].drop_duplicates()

# Merge fuel source into SCADA dataset
scada_with_fuel = scada_data.merge(
    duid_fuel_map,
    on="DUID",
    how="left",
)

# Report any SCADA rows without an associated fuel source
missing_fuel_rows = scada_with_fuel["Fuel Source - Primary"].isna().sum()
print("Rows with missing fuel source:", missing_fuel_rows)

# If there are unmatched DUIDs, print them for debugging
if missing_fuel_rows > 0:
    print(scada_with_fuel.loc[
        scada_with_fuel["Fuel Source - Primary"].isna(),
        "DUID"
    ].drop_duplicates())

# Aggregate SCADA values by timestamp and fuel type, then pivot to columns
dispatch_by_fuel = (
    scada_with_fuel
    .groupby(["SETTLEMENTDATE", "Fuel Source - Primary"], as_index=False)["SCADAVALUE"]
    .sum()
    .pivot(
        index="SETTLEMENTDATE",
        columns="Fuel Source - Primary",
        values="SCADAVALUE",
    )
    .reset_index()
)

# Normalize column names to consistent lowercase identifiers
dispatch_by_fuel = dispatch_by_fuel.rename(columns={
    "Solar": "solar_dispatch",
    "Wind": "wind_dispatch",
    "Hydro": "hydro_dispatch",
})

dispatch_cols = ["solar_dispatch", "wind_dispatch", "hydro_dispatch"]

# Ensure expected columns exist and fill missing pivot columns with zeros
for col in dispatch_cols:
    if col not in dispatch_by_fuel.columns:
        dispatch_by_fuel[col] = 0

# Report and fill any NaNs created by the pivot
print(dispatch_by_fuel[dispatch_cols].isna().sum())
dispatch_by_fuel[dispatch_cols] = dispatch_by_fuel[dispatch_cols].fillna(0)

# Keep only the settlement timestamp plus dispatch columns, sorted chronologically
dispatch_by_fuel = dispatch_by_fuel[
    ["SETTLEMENTDATE"] + dispatch_cols
].sort_values("SETTLEMENTDATE").reset_index(drop=True)

dispatch_by_fuel.columns.name = None

# Preview the result
dispatch_by_fuel.head()

In [ ]:
# Show dataframe info for the aggregated dispatch_by_fuel table
dispatch_by_fuel.info()

# Section 3: Integrating with NSW Price and Demand Data
Load NSW electricity price and demand data, align timestamps with renewable dispatch data, and merge into a single comprehensive dataset for analysis.

In [ ]:
# Save aggregated dispatch_by_fuel to CSV (comment marked 'ignore' in original)
# Use this as a cached export for downstream analysis
dispatch_by_fuel.to_csv("C:\\Users\\Sankalp Roy\\Documents\\BACapstoneProject\\workshop_coding_files\\Scada_2021_2024.csv", index=False)

In [ ]:
# Load NSW price and demand Excel sheet and parse settlement timestamps
df = pd.read_excel(
    r"C:\Users\Sankalp Roy\Documents\BACapstoneProject\workshop_coding_files\NSW Price and Demand 20210701_20240630.xlsx",
    parse_dates=["SETTLEMENTDATE"],
)

# Preview imported table
df.head()

In [ ]:
# Ensure rows are sorted by timestamp for merges and inspection
df = df.sort_values("SETTLEMENTDATE").reset_index(drop=True)

# Preview sorted table
df.head()

In [ ]:
# Quick shape checks to verify row counts between price/demand and dispatch datasets
print(df.shape)
print(dispatch_by_fuel.shape)

In [ ]:
# ------------------------------------------------------------
# 1. Make sure both timestamp columns are real datetime values
# ------------------------------------------------------------

# Convert to pandas datetime for robust comparisons and arithmetic
df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"])
dispatch_by_fuel["SETTLEMENTDATE"] = pd.to_datetime(dispatch_by_fuel["SETTLEMENTDATE"])


# ------------------------------------------------------------
# 2. Basic checks before joining
# ------------------------------------------------------------

print("Rows before join")
print("df:", len(df))
print("dispatch_by_fuel:", len(dispatch_by_fuel))

print("\nDate ranges")
print("df:", df["SETTLEMENTDATE"].min(), "to", df["SETTLEMENTDATE"].max())
print("dispatch_by_fuel:", dispatch_by_fuel["SETTLEMENTDATE"].min(), "to", dispatch_by_fuel["SETTLEMENTDATE"].max())

print("\nDuplicate timestamps")
print("df:", df["SETTLEMENTDATE"].duplicated().sum())
print("dispatch_by_fuel:", dispatch_by_fuel["SETTLEMENTDATE"].duplicated().sum())

print("\nMissing timestamps")
print("df:", df["SETTLEMENTDATE"].isna().sum())
print("dispatch_by_fuel:", dispatch_by_fuel["SETTLEMENTDATE"].isna().sum())


# ------------------------------------------------------------
# 3. Align timestamps
# The price/demand file timestamps are at xx:04:59, xx:09:59, etc.
# SCADA dispatch timestamps are at xx:05:00, xx:10:00, etc.
# Add 1 second to the price/demand timestamps so they align exactly.
# ------------------------------------------------------------

df_join = df.copy()
df_join["SETTLEMENTDATE"] = df_join["SETTLEMENTDATE"] + pd.Timedelta(seconds=1)


# ------------------------------------------------------------
# 4. Check how many timestamps will actually match
# ------------------------------------------------------------

df_times = set(df_join["SETTLEMENTDATE"])
dispatch_times = set(dispatch_by_fuel["SETTLEMENTDATE"])

matching_times = df_times.intersection(dispatch_times)

print("\nTimestamp match check")
print("Matching timestamps:", len(matching_times))
print("df timestamps without dispatch match:", len(df_times - dispatch_times))
print("dispatch timestamps not used by df:", len(dispatch_times - df_times))


# ------------------------------------------------------------
# 5. Left join
# Keeps every row from df_join and adds dispatch columns where timestamps match.
# validate='one_to_one' helps catch accidental duplicate timestamps causing row multiplication.
# ------------------------------------------------------------

merged_df = df_join.merge(
    dispatch_by_fuel,
    on="SETTLEMENTDATE",
    how="left",
    validate="one_to_one",
)

merged_df = merged_df.sort_values("SETTLEMENTDATE").reset_index(drop=True)


# ------------------------------------------------------------
# 6. Checks after joining
# ------------------------------------------------------------

print("\nRows after join")
print("Rows before join:", len(df_join))
print("Rows after join:", len(merged_df))

print("\nMissing joined dispatch values")
print(merged_df[["solar_dispatch", "wind_dispatch", "hydro_dispatch"]].isna().sum())


# ------------------------------------------------------------
# 7. Preview final merged data
# ------------------------------------------------------------

merged_df.head()

In [ ]:
# Save the final merged price/demand + dispatch dataset for analysis
merged_df.to_csv("C:\\Users\\Sankalp Roy\\Documents\\BACapstoneProject\\workshop_coding_files\\NSW_21_24_Price_Demand.csv", index=False)